# 🧬 Sub-tomogram Averaging & 3D Particle Picking for Cryo-ET

### From a reconstructed tomogram to a biological structure

**Hardware Target:** Google Colab, CPU-only (no GPU required)
**Core Libraries:** NumPy, SciPy, scikit-image, scikit-learn
**Prerequisites:** `cryoem/LowSNRNoiseProcessing` (Module 01 — CTF, NCC picking, FRC) and `cryoem/3DCryoEMReconstruction` (Module 02 — Fourier Slice Theorem, back-projection) are strongly recommended first.

---

Modules 01–02 of this phase built the machinery to *reconstruct* a 3-D density from noisy 2-D projections, and Module 03 walked through reconstructing a **real** cryo-ET tomogram in IMOD/Etomo. But a raw tomogram is not a biological answer — it is a noisy, missing-wedge-degraded 3-D image that might contain dozens to thousands of copies of the molecule you actually care about. **Sub-tomogram averaging (STA)** is the step that turns "a tomogram" into "a structure": find every copy of the particle, align them all to a common orientation, and average — exactly the same √N noise-cancellation idea from Module 01's 2-D class averaging, now done in 3-D and complicated by the missing wedge.

This is also one of the most active, CS-heavy areas of cryo-EM/ET research right now (see e.g. RELION, Dynamo, M/Warp, DeepFinder, MemBrain-seg) and is a genuinely useful area to contribute to as a computer scientist — production/commercial tools for the electron microscopy side of this pipeline.

### What You Will Learn
| # | Topic |
|---|---|
| 1 | Why a single tomogram isn't enough — the sub-tomogram averaging idea |
| 2 | The missing wedge in 3-D Fourier space (geometry + code) |
| 3 | Simulating a realistic multi-particle tomogram |
| 4 | 3-D template matching (NCC) for particle picking, with and without an angular search |
| 5 | Why naive averaging destroys structure — the orientation alignment problem |
| 6 | 3-D Fourier Shell Correlation (FSC) — the 3-D analogue of Module 01's FRC |
| 7 | Multi-reference classification of conformational states |
| 8 | How this maps onto production software (RELION / Dynamo / Warp+M) |

**Note:** *For research and educational purposes only.* Everything here is simulated data built to be physically faithful (real missing-wedge geometry, real CTF-free noise model) — it is not a replacement for the production packages linked at the end.

### Install
```bash
pip install -q numpy scipy scikit-image scikit-learn tqdm
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage, fft
from scipy.ndimage import rotate as nd_rotate
from scipy.signal import fftconvolve
from scipy.ndimage import maximum_filter
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor':   '#16213e',
    'axes.edgecolor':   '#e0e0e0',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#e0e0e0',
    'ytick.color':      '#e0e0e0',
    'text.color':       '#e0e0e0',
    'image.cmap':       'inferno',
    'font.size':        11,
})

print('All imports OK')

---
## 1. A 3-D Particle With Two Conformational States

We need a "ground truth" molecule to hide inside a synthetic tomogram. We build a toy hexameric complex — six subunits arranged in a ring around a central hub, the same idea as the ring phantom from Module 01 but now in 3-D. We give it **two conformational states**, `open` and `closed`, by lifting alternating subunits along Z — a stand-in for the real thing biologists look for in cryo-ET (e.g. a channel protein caught in different gating states, or a motor protein at different points in its cycle).

# ── Section Name: Particle phantom ──

In [ ]:
def make_particle_3d(size=24, state='open', n_subunits=6, r_ring=6, r_sub=2.2):
    '''Build a toy hexameric particle phantom in one of two conformational states.'''
    vol = np.zeros((size, size, size))
    c = size // 2
    zz, yy, xx = np.mgrid[0:size, 0:size, 0:size]
    for k in range(n_subunits):
        ang = 2 * np.pi * k / n_subunits
        z_off = 2.5 if (state == 'open' and k % 2 == 0) else 0
        cx = c + r_ring * np.cos(ang)
        cy = c + r_ring * np.sin(ang)
        cz = c + z_off
        vol += np.exp(-((xx-cx)**2 + (yy-cy)**2 + (zz-cz)**2) / (2*r_sub**2))
    vol += 0.6*np.exp(-((xx-c)**2 + (yy-c)**2 + (zz-c)**2) / (2*(r_sub*1.2)**2))
    return vol / vol.max()

BOX = 24
vol_open   = make_particle_3d(BOX, state='open')
vol_closed = make_particle_3d(BOX, state='closed')

fig, axes = plt.subplots(1, 4, figsize=(13, 3.3))
axes[0].imshow(vol_open.max(axis=0));   axes[0].set_title('open — XY max-proj')
axes[1].imshow(vol_open.max(axis=1));   axes[1].set_title('open — XZ max-proj')
axes[2].imshow(vol_closed.max(axis=0)); axes[2].set_title('closed — XY max-proj')
axes[3].imshow(vol_closed.max(axis=1)); axes[3].set_title('closed — XZ max-proj')
for ax in axes: ax.axis('off')
plt.suptitle('Two conformational states of the same particle (max-intensity projections)', y=1.05)
plt.tight_layout(); plt.show()

print(f'Voxel-wise difference between states (L2 norm): {np.linalg.norm(vol_open - vol_closed):.2f}')

---
## 2. The Missing Wedge in 3-D Fourier Space

A single-axis tilt series only rotates the specimen through a limited range, typically $\pm60^\circ$ about the tilt (Y) axis, because at higher tilts the effective sample thickness along the beam grows as $1/\cos\theta$ and the image becomes too absorbed/blurred to use. Each tilt image contributes one central *plane* of the 3-D Fourier transform (the Central Slice Theorem from Module 02, now applied plane-by-plane instead of line-by-line). As the tilt sweeps $\theta \in [-\theta_{max}, \theta_{max}]$, the union of all these planes leaves a **wedge-shaped cone of missing frequencies** around the beam (Z) axis, with half-angle

$$\phi_{missing} = 90^\circ - \theta_{max}$$

For $\theta_{max}=60^\circ$ this cone is $30^\circ$ wide, which removes roughly a third of all Fourier information — exactly the "coverage fraction" printed below. The direct real-space consequence is that objects appear **elongated along Z** (the exact artefact you're asked to identify in Module 03's diagnostic questions) and any operation that averages in Fourier space (like sub-tomogram averaging, below) needs a wedge-aware weighting or the missing region will bias the average.

# ── Section Name: Missing wedge geometry ──

In [ ]:
def missing_wedge_mask(size, max_tilt_deg=60):
    '''True = sampled frequency, False = inside the missing-wedge cone around the beam (Z) axis.'''
    zz, yy, xx = np.mgrid[-size//2:size//2, -size//2:size//2, -size//2:size//2]
    angle = np.degrees(np.arctan2(xx, zz + 1e-8))            # angle from +Z axis
    d_from_z = np.minimum(np.abs(angle), 180 - np.abs(angle))
    missing = d_from_z <= (90 - max_tilt_deg)
    keep = ~missing
    keep |= (xx**2 + yy**2 + zz**2) < 2                       # always keep the DC term
    return keep

def apply_missing_wedge(vol, max_tilt_deg=60):
    F = fft.fftshift(fft.fftn(vol))
    keep = missing_wedge_mask(vol.shape[0], max_tilt_deg)
    return np.real(fft.ifftn(fft.ifftshift(F * keep))), keep

MAX_TILT = 60
vol_wedge, keep_mask = apply_missing_wedge(vol_open, MAX_TILT)
print(f'Fourier coverage at +-{MAX_TILT} deg tilt: {keep_mask.mean()*100:.1f}%  '
      f'(missing wedge removes ~{(1-keep_mask.mean())*100:.1f}%)')

fig, axes = plt.subplots(1, 3, figsize=(11, 3.3))
axes[0].imshow(keep_mask[:, keep_mask.shape[1]//2, :]); axes[0].set_title('Sampled region (kx-kz slice)')
axes[1].imshow(vol_open.max(axis=1));  axes[1].set_title('Before wedge — XZ proj')
axes[2].imshow(vol_wedge.max(axis=1)); axes[2].set_title('After wedge — XZ proj (Z-elongated)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

---
## 3. Simulating a Multi-Particle Tomogram

We now place several copies of the particle (alternating `open`/`closed`, random 3-D orientation, random position with a minimum separation) into an empty volume, apply the missing wedge, and add detector noise — the same forward-model philosophy as Module 01's 2-D phantom, extended to 3-D and to *multiple* particles per field of view, which is what a real tomogram looks like.

# ── Section Name: Tomogram simulation ──

In [ ]:
def build_tomogram(n_particles=8, tomo_size=110, box=24, noise_sigma=0.4,
                    wedge_deg=60, rng=None):
    '''Place n_particles copies (alternating conformational state, random pose) into a tomogram.'''
    rng = rng or np.random.default_rng(0)
    tomo = np.zeros((tomo_size,)*3)
    gt, centers = [], []
    min_sep = int(box * 1.3)
    for i in range(n_particles):
        state = 'open' if i % 2 == 0 else 'closed'
        base = make_particle_3d(size=box, state=state)
        angles = rng.uniform(0, 360, size=3)
        rotated = base.copy()
        for ax, ang in zip([(0,1), (0,2), (1,2)], angles):
            rotated = nd_rotate(rotated, ang, axes=ax, reshape=False, order=1, mode='constant')
        margin = box
        for _ in range(200):
            cz, cy, cx = rng.integers(margin, tomo_size-margin, size=3)
            if all(np.linalg.norm(np.array([cz,cy,cx]) - np.array(c)) > min_sep for c in centers):
                break
        centers.append((cz, cy, cx))
        s = box // 2
        tomo[cz-s:cz+s, cy-s:cy+s, cx-s:cx+s] += rotated
        gt.append((cz, cy, cx, angles, state))
    tomo_wedge, _ = apply_missing_wedge(tomo, wedge_deg)
    tomo_noisy = tomo_wedge + rng.normal(0, noise_sigma, tomo.shape)
    return tomo_noisy, gt

TOMO_SIZE, N_PARTICLES = 110, 8
tomo, gt = build_tomogram(n_particles=N_PARTICLES, tomo_size=TOMO_SIZE, box=BOX,
                           noise_sigma=0.4, wedge_deg=MAX_TILT, rng=np.random.default_rng(1))

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
mid = TOMO_SIZE // 2
axes[0].imshow(tomo[mid]);        axes[0].set_title('Central XY slice')
axes[1].imshow(tomo.max(axis=0)); axes[1].set_title('XY max-projection\n(particles visible)')
axes[2].imshow(tomo.max(axis=1)); axes[2].set_title('XZ max-projection')
for ax in axes: ax.axis('off')
plt.suptitle(f'Simulated tomogram — {N_PARTICLES} particles, wedge=+-{MAX_TILT} deg, noise sigma=0.4', y=1.02)
plt.tight_layout(); plt.show()

print(f'Ground truth: {[g[4] for g in gt]}')

---
## 4. 3-D Particle Picking by Template Matching

Exactly like Module 01's 2-D NCC picking, but the correlation is now computed over a 3-D template and volume. The complication that's new in 3-D: **the template must be tried at many orientations**, because a randomly-oriented particle will not match a single fixed-orientation template well. We compare single-orientation picking against a coarse angular search that takes the best score over several random rotations of the template — this is a simplified version of what production pickers (crYOLO, Warp's BoxNet, template-matching in Dynamo/PyTom) all have to solve.

$$\text{NCC}(\mathbf{r}) = \frac{\sum_{\mathbf{u}} \big(V(\mathbf{r}+\mathbf{u}) - \bar{V}\big)\, T(\mathbf{u})}{\sqrt{\sum_{\mathbf{u}} \big(V(\mathbf{r}+\mathbf{u}) - \bar{V}\big)^2} \cdot \|T\|}$$

# ── Section Name: 3D template matching ──

In [ ]:
def normalized_cross_corr_3d(vol, template):
    '''3-D NCC via FFT-based correlation (fast enough for Colab CPU).'''
    t = template - template.mean()
    t = t / (np.linalg.norm(t) + 1e-8)
    corr = fftconvolve(vol, t[::-1, ::-1, ::-1], mode='same')
    local_energy = np.sqrt(ndimage.uniform_filter(vol**2, size=template.shape) + 1e-6)
    return corr / (local_energy * np.sqrt(np.prod(template.shape)) + 1e-6)

def angular_search_ncc(vol, template, n_angles=12, seed=0):
    '''Take the best NCC score over several random template orientations.'''
    rng = np.random.default_rng(seed)
    best = None
    for i in range(n_angles):
        ang = rng.uniform(0, 360, size=3) if i > 0 else np.zeros(3)
        t = template.copy()
        for ax, a in zip([(0,1), (0,2), (1,2)], ang):
            t = nd_rotate(t, a, axes=ax, reshape=False, order=1, mode='constant')
        score = normalized_cross_corr_3d(vol, t)
        best = score if best is None else np.maximum(best, score)
    return best

def find_peaks_3d(score, box, n_peaks, min_dist=None):
    min_dist = min_dist or box
    local_max = (maximum_filter(score, size=min_dist) == score)
    coords = np.argwhere(local_max)
    vals = score[local_max]
    return coords[np.argsort(-vals)][:n_peaks]

def recall_at(score_map, gt_centers, box, n_peaks, min_dist, tol=8):
    peaks = find_peaks_3d(score_map, box, n_peaks, min_dist)
    hit = sum(1 for p in peaks if np.linalg.norm(gt_centers - p, axis=1).min() < tol)
    return hit, peaks

ref_template = make_particle_3d(size=BOX, state='open')
gt_centers = np.array([[g[0], g[1], g[2]] for g in gt])

ncc_single = normalized_cross_corr_3d(tomo, ref_template)
ncc_search = angular_search_ncc(tomo, ref_template, n_angles=12)

r_single, peaks_single = recall_at(ncc_single, gt_centers, BOX, N_PARTICLES, min_dist=int(BOX*1.3))
r_search, peaks_search = recall_at(ncc_search, gt_centers, BOX, N_PARTICLES, min_dist=int(BOX*1.3))

print(f'Single fixed-orientation template  -> recall {r_single}/{N_PARTICLES}')
print(f'Angular-search template matching   -> recall {r_search}/{N_PARTICLES}')

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.imshow(tomo.max(axis=0))
ax.scatter(gt_centers[:,2], gt_centers[:,1], s=140, facecolors='none', edgecolors='#00ff88', linewidths=2, label='ground truth')
ax.scatter(peaks_search[:,2], peaks_search[:,1], s=60, marker='x', c='#ff3366', label='angular-search picks')
ax.legend(loc='upper right', fontsize=8); ax.axis('off')
ax.set_title('Picked particle locations (XY max-projection)')
plt.tight_layout(); plt.show()

---
## 5. Sub-tomogram Extraction — Why Naive Averaging Fails

We now crop a small box around each picked location (a *sub-tomogram*) and try the simplest possible thing: average them all together. This is exactly Module 01's 2-D class-averaging idea — but it silently assumes every extracted box already shows the particle in the **same orientation**. In a real tomogram the particles are randomly rotated, so a naive average blurs the six-fold ring structure into a featureless blob: rotational misalignment destroys signal exactly the way translational misalignment would.

The fix is **alignment before averaging**: estimate each particle's orientation (against a reference) and rotate it back to a common frame first. We compare three cases:

1. **Naive** — average with no alignment at all
2. **Ground-truth aligned** — an oracle that knows the true rotation used to generate each particle (upper bound)
3. **Estimated aligned** — a realistic angular search against the reference template, no oracle information

# ── Section Name: Orientation alignment ──

In [ ]:
def extract_subvolume(vol, center, box):
    z, y, x = center
    s = box // 2
    z0, y0, x0 = int(z-s), int(y-s), int(x-s)
    return vol[z0:z0+box, y0:y0+box, x0:x0+box]

def unrotate(vol, angles):
    out = vol.copy()
    for ax, ang in zip([(1,2), (0,2), (0,1)], angles[::-1]):
        out = nd_rotate(out, -ang, axes=ax, reshape=False, order=1, mode='constant')
    return out

def estimate_rotation(subvol, template, n_candidates=40, seed=0):
    '''Coarse angular search: which candidate rotation of the template best matches this subvolume?'''
    rng = np.random.default_rng(seed)
    best_score, best_angles = -np.inf, np.zeros(3)
    for i in range(n_candidates):
        ang = rng.uniform(0, 360, size=3) if i > 0 else np.zeros(3)
        t = template.copy()
        for ax, a in zip([(0,1), (0,2), (1,2)], ang):
            t = nd_rotate(t, a, axes=ax, reshape=False, order=1, mode='constant')
        score = np.sum(t * subvol)
        if score > best_score:
            best_score, best_angles = score, ang
    return best_angles

subvols = [extract_subvolume(tomo, (g[0], g[1], g[2]), BOX) for g in gt]

naive_avg = np.mean(subvols, axis=0)
gt_aligned = [unrotate(sv, g[3]) for sv, g in zip(subvols, gt)]
gt_avg = np.mean(gt_aligned, axis=0)

est_aligned = []
for sv in subvols:
    ang = estimate_rotation(sv, ref_template, n_candidates=40)
    sv_aligned = sv.copy()
    for ax, a in zip([(1,2), (0,2), (0,1)], ang[::-1]):
        sv_aligned = nd_rotate(sv_aligned, -a, axes=ax, reshape=False, order=1, mode='constant')
    est_aligned.append(sv_aligned)
est_avg = np.mean(est_aligned, axis=0)

clean_ref = make_particle_3d(size=BOX, state='open')
def corr_to_ref(vol, ref):
    return np.corrcoef(vol.flatten(), ref.flatten())[0, 1]

results = {
    'single particle (noisy)': corr_to_ref(subvols[0], clean_ref),
    'naive average (unaligned)': corr_to_ref(naive_avg, clean_ref),
    'ground-truth aligned average': corr_to_ref(gt_avg, clean_ref),
    'estimated aligned average': corr_to_ref(est_avg, clean_ref),
}
for k, v in results.items():
    print(f'{k:32s} correlation to clean reference = {v:.3f}')

fig, axes = plt.subplots(1, 4, figsize=(14, 3.3))
for ax, (title, vol) in zip(axes, [('single particle', subvols[0]), ('naive avg (unaligned)', naive_avg),
                                     ('ground-truth aligned', gt_avg), ('estimated aligned', est_avg)]):
    ax.imshow(vol.max(axis=0)); ax.set_title(title, fontsize=10); ax.axis('off')
plt.suptitle('Naive averaging blurs the ring structure — alignment recovers it', y=1.05)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(6, 3.5))
labels_plot = list(results.keys())[1:]
vals_plot = [results[k] for k in labels_plot]
ax.bar(range(len(vals_plot)), vals_plot, color=['#ff3366', '#00ff88', '#3388ff'])
ax.set_xticks(range(len(vals_plot))); ax.set_xticklabels(['naive', 'gt-aligned\n(oracle)', 'estimated\naligned'], fontsize=9)
ax.set_ylabel('correlation to clean reference')
ax.set_title('Alignment recovers the signal averaging alone destroys')
plt.tight_layout(); plt.show()

---
## 6. Resolution: 3-D Fourier Shell Correlation (FSC)

Module 01 introduced **Fourier Ring Correlation (FRC)** to measure 2-D class-average resolution by splitting particles into two halves, averaging each independently, and correlating shell-by-shell in Fourier space. **FSC is the direct 3-D generalisation**: instead of rings we correlate over spherical shells of radius $r$,

$$\text{FSC}(r) = \frac{\sum_{\mathbf{k} \in \text{shell}(r)} F_1(\mathbf{k})\, F_2^*(\mathbf{k})}{\sqrt{\sum_{\mathbf{k} \in \text{shell}(r)} |F_1(\mathbf{k})|^2 \sum_{\mathbf{k} \in \text{shell}(r)} |F_2(\mathbf{k})|^2}}$$

and the same **0.143 half-bit criterion** from Module 01 marks the resolution at which the two independent averages stop agreeing — i.e. where noise starts to dominate signal. This is the standard resolution metric reported by RELION, cryoSPARC, and every production STA package.

# ── Section Name: 3D Fourier Shell Correlation ──

In [ ]:
def fsc(vol1, vol2, n_shells=12):
    F1 = fft.fftshift(fft.fftn(vol1))
    F2 = fft.fftshift(fft.fftn(vol2))
    size = vol1.shape[0]
    zz, yy, xx = np.mgrid[-size//2:size//2, -size//2:size//2, -size//2:size//2]
    r = np.sqrt(xx**2 + yy**2 + zz**2)
    shells = np.linspace(0, size//2, n_shells+1)
    vals = []
    for i in range(n_shells):
        mask = (r >= shells[i]) & (r < shells[i+1])
        num = np.sum(F1[mask] * np.conj(F2[mask])).real
        den = np.sqrt(np.sum(np.abs(F1[mask])**2) * np.sum(np.abs(F2[mask])**2))
        vals.append(num / (den + 1e-8))
    return np.array(vals), (shells[:-1] + shells[1:]) / 2

half1 = np.mean(est_aligned[0::2], axis=0)
half2 = np.mean(est_aligned[1::2], axis=0)
fsc_vals, radii = fsc(half1, half2)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(radii, fsc_vals, marker='o', color='#00ff88')
ax.axhline(0.143, color='#ff3366', linestyle='--', label='0.143 half-bit threshold')
crossing = radii[np.argmax(fsc_vals < 0.143)] if np.any(fsc_vals < 0.143) else radii[-1]
ax.axvline(crossing, color='#e0e0e0', linestyle=':', alpha=0.6)
ax.set_xlabel('spatial frequency (Fourier voxels from center)')
ax.set_ylabel('FSC')
ax.set_title(f'Half-set FSC of the aligned average (N={len(est_aligned)} particles)')
ax.legend(); plt.tight_layout(); plt.show()

print(f'FSC crosses 0.143 at radius ~{crossing:.1f} voxels — with only {len(est_aligned)} particles, '
      f'resolution is modest; real STA runs use hundreds to thousands of particles to push this further out.')

---
## 7. Classifying Conformational States (Multi-Reference Alignment)

Real biological samples are usually mixtures — some fraction of particles in one conformation, some in another — and averaging them all together would blur the two states into an uninterpretable mush (the same lesson as Section 5, but along a different axis: conformational heterogeneity rather than orientation). The standard fix is **multi-reference classification**: score every aligned sub-tomogram against each candidate reference and assign it to whichever it matches best (this is a simplified stand-in for RELION/Dynamo's maximum-likelihood 3-D classification).

# ── Section Name: Multi-reference classification ──

In [ ]:
ref_open, ref_closed = make_particle_3d(BOX, 'open'), make_particle_3d(BOX, 'closed')
true_bin = np.array([0 if g[4] == 'open' else 1 for g in gt])

def classify(subvol_list, ref_a, ref_b):
    preds = []
    for sv in subvol_list:
        s_a = np.corrcoef(sv.flatten(), ref_a.flatten())[0, 1]
        s_b = np.corrcoef(sv.flatten(), ref_b.flatten())[0, 1]
        preds.append(0 if s_a > s_b else 1)
    return np.array(preds)

def acc(preds, true):
    return max((preds == true).mean(), (preds != true).mean())  # allow label-swap

pred_unaligned = classify(subvols, ref_open, ref_closed)
pred_aligned   = classify(est_aligned, ref_open, ref_closed)

acc_unaligned, acc_aligned = acc(pred_unaligned, true_bin), acc(pred_aligned, true_bin)
print('true states:      ', [g[4] for g in gt])
print(f'classify on RAW (unaligned) subvolumes    -> accuracy = {acc_unaligned:.2f}')
print(f'classify on ALIGNED subvolumes            -> accuracy = {acc_aligned:.2f}')
print()
if acc_aligned > acc_unaligned:
    print(f'Alignment alone recovered {acc_aligned - acc_unaligned:.2f} of classification accuracy — comparing')
    print('conformations only makes sense once every particle is in the same reference frame.')
else:
    print('With only a handful of particles this is a noisy problem — re-run with a different seed')
    print('or a larger N_PARTICLES (Section 3) and this comparison will vary run to run.')
print('Real cryo-ET 3D classification runs typically use hundreds to thousands of sub-tomograms per')
print('class for a statistically reliable separation — try raising N_PARTICLES above and re-running.')

---
## 8. From This Notebook to Production Software

Every step above has a direct production equivalent, the same "computational concept → real tool" mapping used in Module 03's readme:

| What we built here | Production tool that does this |
|---|---|
| Missing-wedge mask | RELION / Warp CTF+wedge weighting |
| 3-D NCC template matching | PyTom, Dynamo template matching, STOPGAP |
| Angular-search picking | crYOLO, Warp's BoxNet (deep-learning pickers, trained rather than searched) |
| Alignment before averaging | RELION `relion_refine`, Dynamo `dalign`, M (Warp's STA refinement) |
| 3-D FSC / 0.143 criterion | RELION post-processing, cryoSPARC |
| Multi-reference classification | RELION 3-D Classification, Dynamo MRA |
| Membrane / organelle segmentation in tomograms (not covered here) | MemBrain-seg, DeepFinder |

If you want to push this further with a real dataset, EMPIAR-10164 (the same *Mycoplasma pneumoniae* dataset from Module 03) has published sub-tomogram averaging results you can compare against once you have a reconstructed tomogram.

---
## Key Takeaways

| Concept | Key Insight |
|---|---|
| **Missing wedge** | A $\pm60^\circ$ tilt range leaves a $30^\circ$ cone of Fourier space unsampled — about a third of all information |
| **3-D picking** | Fixed-orientation template matching misses randomly-rotated particles; an angular search is required |
| **Alignment before averaging** | Averaging misaligned particles destroys structure just as surely as misaligned positions do |
| **FSC / 0.143** | The direct 3-D generalisation of Module 01's FRC — same criterion, spherical shells instead of rings |
| **Classification** | Conformational heterogeneity needs multi-reference classification, not a single global average, and needs real sample sizes to be reliable |

---
## Questions / Discussion

1. Why does the missing-wedge cone sit around the **beam (Z) axis** rather than, say, the tilt axis (Y)? What real-space artefact does this cause, and how does it connect to the "Z-elongation" diagnostic question in Module 03?
2. The angular search in Section 4/5 tries `n_candidates` *random* orientations. What would change if you used a systematic grid over Euler angles instead — what's the trade-off against runtime?
3. In Section 6 we split particles into two halves by simple alternation (`[0::2]`, `[1::2]`). Would splitting differently (e.g. spatially, by tomogram region) change the FSC in a way that reveals a different kind of error?
4. Classification accuracy in Section 7 was modest with N=8 particles. Sketch (in words or code) how you'd expect accuracy to scale with N, and why real STA pipelines report classification results only after reaching hundreds of particles per class.

## Bonus — Going Further

- **Increase `N_PARTICLES`** and re-run Sections 5–7 — sub-tomogram averaging is fundamentally a "more copies = better SNR" method (√N scaling, same as Module 01), so the gap between naive and aligned averaging, and the classification accuracy, should both visibly improve.
- **Replace the angular search** in `estimate_rotation` with a coarse-to-fine search (wide random search, then refine locally around the best candidate) and measure how much faster it converges to a good alignment.
- **Try a 3rd conformational state** — extend `make_particle_3d` with an intermediate state and `classify()` with `n_clusters=3`, and see how classification accuracy degrades as classes get closer together.
- **Wedge-aware averaging** — right now `naive_avg`/`gt_avg`/`est_avg` just average voxels directly, which under-weights regions where more particles happen to have wedge coverage. Try building a per-voxel weight volume (count of particles with real coverage there) and dividing by it instead of a flat average — this is what real STA software does.